# Extração dos três modelos contextuais
Este notebook expõe a etapa de extração descrita no artigo, usando o apoio de03/11 preservado. Ele não treina XGBoost nem substitui as matrizes originais. Para repetir a classificação publicada, use TG_Robustez_Seis_Representacoes.ipynb e os vetores congelados fornecidos. Uma nova extração pode apresentar diferenças numéricas de plataforma/precisão; registre versões e confira antes de substituir resultados. A extração integral não foi repetida nesta revisão de10/11.

In [ ]:
%pip install -q numpy==2.2.6 pandas==2.2.3 transformers==4.52.4 tokenizers==0.21.4 tqdm==4.67.1
# Usa o PyTorch do ambiente Colab; a célula seguinte registra sua versão.

## Escolher arquivos e modelo
Extraia os quatro arquivos de reprodução na mesma pasta. Escolha uma base e um modelo por execução; para extração use GPU no Colab, quando disponível. O padrão abaixo não inicia a extração. A variável EXECUTAR_EXTRACAO deve ser alterada conscientemente. Os checkpoints ficam em vetores_novos, separados das matrizes usadas no artigo.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, importlib.metadata as im
PASTA = Path(os.environ.get('TG_RAIZ','/content/TG_REPRODUCAO_ROBUSTEZ'))
BASE = 'faketrue'
METODO = 'bertimbau'  # bertimbau, e5 ou qwen3
EXECUTAR_EXTRACAO = False
os.environ['TG_RAIZ'] = str(PASTA)
import subprocess
CODIGO = Path.cwd()
if not (CODIGO / 'protocolo_robustez_20261106.py').is_file():
    CODIGO = Path('/content/tg-desinformacao')
    if not CODIGO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/diego-henrique-vaz/avaliacao-de-representacoes-vetoriais-na-identificacao-de-desinformacao', str(CODIGO)], check=True)
sys.path.insert(0, str(CODIGO))
import pandas as pd
from apoio_contextuais_simples import extrair_embeddings
arquivo = PASTA/'robustez_privada_2026_11_06/dados'/f'{BASE}.csv'
entradas = json.loads((PASTA/'robustez_privada_2026_11_06/dados/ENTRADAS_FIXAS.json').read_text(encoding='utf8'))
assert hashlib.sha256(arquivo.read_bytes()).hexdigest()==entradas['csv_sha256'][BASE]
dados = pd.read_csv(arquivo,keep_default_na=False,dtype={'doc_id':str})
spec = json.loads((PASTA/'experimentos_divisoes_2026_11_05/vetores_contextuais'/BASE/METODO/'spec.json').read_text(encoding='utf8'))
assert spec['name']==METODO and spec['max_length']==512
modelo = spec['model']
versoes = {p:im.version(p) for p in ['numpy','pandas','torch','transformers','tokenizers','tqdm']}
print('Versões desta execução:',versoes)
print('Configuração original:',modelo)

## Extrair e preservar uma nova matriz
O apoio divide cada notícia inteira em blocos, aplica as máscaras e a agregação especificadas e mantém um checkpoint a cada16 documentos. BERTimbau usa média do conteúdo; E5 usa média dos tokens válidos e prefixo query; Qwen3 usa o último token válido e a instrução fixa registrada em spec.json. Nenhum rótulo é fornecido ao encoder.

In [ ]:
if EXECUTAR_EXTRACAO:
    saida = PASTA/'vetores_novos'/BASE
    saida.mkdir(parents=True,exist_ok=True)
    (saida/f'{METODO}_versoes.json').write_text(json.dumps(versoes,indent=2),encoding='utf8')
    vetores = extrair_embeddings(dados.text.tolist(),identificador=modelo['id'],
        revisao=modelo['revision'],metodo=METODO,dimensao=modelo['dim'],
        prefixo=modelo.get('prefix',''),pasta=saida,
        lote=2 if METODO=='qwen3' and BASE in ['fakebr','boatosbr'] else 4,limite=512)
    print('Matriz nova:',vetores.shape,'| pasta:',saida)
else:
    print('Extração desativada. Para os resultados publicados, use os vetores congelados.')

## Preservação
Primeiro backup histórico + três snapshots completos mais recentes, validados antes da rotação. Vetores novos e checkpoints ativos ficam fora da exclusão rotativa. O espaço /content do Colab é temporário; use uma pasta persistente já autorizada ou baixe os arquivos ao concluir. Este notebook não monta Drive automaticamente nem publica resultados.